## Installation (execute only if running on a cloud platform!)¶

In [5]:
# -- Use the following line for google colab removing the hash at the beginning.
! pip install -q 'corner==2.2.2' 'bilby==2.2.2' 'astropy==6.0.1'

## Initialization

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import corner

In [2]:
!pip install healpy astropy pyvo pandas

In [3]:
# REQUIREMENTS (already used in your notebook):
# !pip install pyvo astropy healpy pandas

import json, numpy as np, pandas as pd, healpy as hp
from astropy.io import fits
from astropy.cosmology import Planck18 as cosmo
from astropy import units as u

# ---------- Config ----------
TAP_URL = "https://datalab.noirlab.edu/tap"
SKYMAP_URL = "https://dcc.ligo.org/LIGO-P2000230/public/GW190814_skymap.fits.gz"
CREDIBLE = 0.90
SQL_FATAL_QUALITY = True          # keep True to avoid multi-million row downloads
MAG_MIN, MAG_MAX = 16.0, 24.5     # light magnitude guard for tractability
NSIDE_FORCE = None                # set to an int to override skymap NSIDE if needed (usually None)
NEST_FORCE  = None                # set to True/False to override, else read from skymap

Z_MIN, Z_MAX = 0.043, 0.061       # from GW distance ~ 200–280 Mpc

from pyvo import dal as vo

svc = vo.TAPService(TAP_URL)

# ---------- Helpers ----------
def hpd_threshold(prob, level):
    flat = prob.ravel()
    order = np.argsort(flat)[::-1]
    csum = np.cumsum(flat[order])
    return flat[order[np.searchsorted(csum, level * flat.sum())]]

def healpix_components(mask, nside, nest=True):
    npix = mask.size
    visited = np.zeros(npix, bool)
    comps = []
    idxs = np.where(mask)[0]
    idxset = set(idxs.tolist())
    for s in idxs:
        if visited[s]: continue
        q=[int(s)]; visited[s]=True; comp=[int(s)]
        while q:
            p=q.pop()
            neigh = hp.get_all_neighbours(nside, p, nest=nest)
            for nb in neigh[neigh>=0]:
                if (nb in idxset) and (not visited[nb]):
                    visited[nb]=True; q.append(int(nb)); comp.append(int(nb))
        comps.append(comp)
    return comps

def minimal_ra_span(ra_deg):
    ra = np.mod(ra_deg, 360.0)
    s = np.sort(ra); dbl = np.concatenate([s, s+360])
    N=len(s); best=(1e9,0,0)
    for i in range(N):
        j=i+N-1
        w=dbl[j]-dbl[i]
        if w<best[0]:
            a=(dbl[i])%360.0; b=(a+w)%360.0; best=(w,a,b)
    return best[1], best[2]

# ---------- 1) Load skymap & build 90% main island ----------
with fits.open(SKYMAP_URL, memmap=True) as h:
    tab = h[1].data; hdr = h[1].header
    prob      = np.asarray(tab["PROB"], float)
    distmu    = np.asarray(tab["DISTMU"], float)
    distsigma = np.asarray(tab["DISTSIGMA"], float)
    nside = int(hdr["NSIDE"]) if NSIDE_FORCE is None else int(NSIDE_FORCE)
    nest  = hdr.get("ORDERING","NESTED").upper().startswith("NEST") if NEST_FORCE is None else bool(NEST_FORCE)

thr  = hpd_threshold(prob, CREDIBLE)
mask = prob >= thr
comps = healpix_components(mask, nside, nest=nest)
main  = np.array(comps[np.argmax([prob[c].sum() for c in comps])], dtype=int)

theta, phi = hp.pix2ang(nside, main, nest=nest)
dec_deg = 90.0 - np.degrees(theta)
ra_deg  = np.degrees(phi) % 360.0
dec_min, dec_max = float(dec_deg.min()), float(dec_deg.max())
ra_min, ra_max   = minimal_ra_span(ra_deg)

funnel = {}

# ---------- 2) SQL rectangle query (DES main ⨝ y6_gold), fatal-only limits ----------
# We *do not* apply morphology or photo-z in SQL to preserve the funnel order.
where_sky = f"""
(g.ra BETWEEN {ra_min} AND {ra_max}) AND
(g.dec BETWEEN {dec_min} AND {dec_max})
"""

where_quality = "1=1"
if SQL_FATAL_QUALITY:
    where_quality = f"(g.flags_i < 4) AND (g.mag_auto_i BETWEEN {MAG_MIN} AND {MAG_MAX})"

adql = f"""
SELECT
  g.coadd_object_id AS coadd_object_id,
  g.ra, g.dec,
  g.mag_auto_i, g.flags_i,
  y.dnf_z,
  y.wavg_spread_model_z, y.wavg_spreaderr_model_z,
  y.spread_model_z,      y.spreaderr_model_z
FROM des_dr2.main AS g
JOIN des_dr2.y6_gold AS y
  ON g.coadd_object_id = y.coadd_object_id
WHERE
  {where_sky}
  AND {where_quality}
"""

rect_tbl = svc.search(adql).to_table()
funnel["step1_rectangle_sql"] = int(len(rect_tbl))

print(f"[1] Rectangle (SQL) — rows: {funnel['step1_rectangle_sql']}  "
      f"[flags_i<4={SQL_FATAL_QUALITY}, mag {MAG_MIN}–{MAG_MAX}]")

# ---------- 3) Exact 90% main-island mask ----------
theta = np.radians(90.0 - np.array(rect_tbl["dec"]))
phi   = np.radians(np.array(rect_tbl["ra"]) % 360.0)
pix   = hp.ang2pix(nside, theta, phi, nest=nest)
in_island = np.isin(pix, main)
island_tbl = rect_tbl[in_island]
funnel["step2_inside_island"] = int(len(island_tbl))
print(f"[2] Inside exact 90% main island — rows: {funnel['step2_inside_island']}")

# ---------- 4) Morphology (galaxy-like) ----------
wok = (island_tbl["wavg_spread_model_z"] > -98) & (island_tbl["wavg_spreaderr_model_z"] > -98)
m3  = np.where(
    wok,
    island_tbl["wavg_spread_model_z"] + 3*island_tbl["wavg_spreaderr_model_z"],
    island_tbl["spread_model_z"]      + 3*island_tbl["spreaderr_model_z"]
)
gal_mask = m3 > 0.005
gal_tbl  = island_tbl[gal_mask]
funnel["step3_morph_galaxies"] = int(len(gal_tbl))
print(f"[3] Morphology (extended) — rows: {funnel['step3_morph_galaxies']}")

# ---------- 5) Photo-z window (distance slice) ----------
z_ok = (gal_tbl["dnf_z"] >= Z_MIN) & (gal_tbl["dnf_z"] <= Z_MAX)
final_tbl = gal_tbl[z_ok]
funnel["step4_photoz_window"] = int(len(final_tbl))
print(f"[4] Photo-z in [{Z_MIN},{Z_MAX}] — rows: {funnel['step4_photoz_window']}")

# ---------- Save artifacts ----------
# CSV + FITS of final candidates
final_df = final_tbl.to_pandas()
final_df.to_csv("GW190814_candidates_90pct.csv", index=False)
final_tbl.write("GW190814_candidates_90pct.fits", overwrite=True)

# Counts JSON + metadata
meta = {
    "event": "GW190814",
    "credible_level": CREDIBLE,
    "main_island_prob_mass": float(prob[main].sum()),
    "nside": nside, "nest": nest,
    "ra_min_deg": ra_min, "ra_max_deg": ra_max,
    "dec_min_deg": dec_min, "dec_max_deg": dec_max,
    "sql_fatal_quality": SQL_FATAL_QUALITY,
    "mag_range_i": [MAG_MIN, MAG_MAX],
    "z_window": [Z_MIN, Z_MAX]
}
out = {"counts": funnel, "meta": meta}
with open("GW190814_funnel_counts.json", "w") as f:
    json.dump(out, f, indent=2)

print("\nSaved:")
print(" - GW190814_candidates_90pct.csv")
print(" - GW190814_candidates_90pct.fits")
print(" - GW190814_funnel_counts.json")


[1] Rectangle (SQL) — rows: 2119644  [flags_i<4=True, mag 16.0–24.5]
[2] Inside exact 90% main island — rows: 1393743


[3] Morphology (extended) — rows: 1321738
[4] Photo-z in [0.043,0.061] — rows: 589

Saved:
 - GW190814_candidates_90pct.csv
 - GW190814_candidates_90pct.fits
 - GW190814_funnel_counts.json
